# Imports


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from datetime import date as date
import yfinance as yf
import re
from hmmlearn.hmm import GaussianHMM
import matplotlib.pyplot as plt

# Data Management

In [ ]:
# Data extraction
start_date = '2000-01-01'
end_date = date.today()
symbol = 'HBAR-USD'
data = yf.download(symbol, start_date, end_date)
data

In [ ]:
print(data)

In [ ]:
data.head(10)

In [ ]:
data.tail(10)

In [ ]:
data.describe()

In [ ]:
data.info()

In [ ]:
def unify_column_names(name: str) -> str:
    """Converts column names to lowercase, replaces spaces/hyphens with underscores, and handles MultiIndex."""
    return (
        tuple(re.sub(r"[-\s]+", "_", str(n).lower()) for n in name)
        if isinstance(name, tuple)
        else re.sub(r"[-\s]+", "_", str(name).lower())
    )


# Apply transformation to column names
data.columns = data.columns.map(unify_column_names)

In [ ]:
print(f"Unified columns names: \n{data.columns}")

In [ ]:
# Add returns and range
df = data.copy()
df["returns"] = df["close"].pct_change()
df["range"] = (df["high"] / df["low"]) - 1

In [ ]:
df.head(10)

In [ ]:
# Determine the total of NA values and drop them
na_count = df.isna().sum().sum()
if na_count > 0:
    print(f"Dropping {na_count} NA value(s)")
    df.dropna(inplace=True)

print(f"Length: {len(df)}")
df.head(10)

In [ ]:
# Add moving averages
df["ma_12"] = df["close"].rolling(window=12).mean()
df["ma_21"] = df["close"].rolling(window=21).mean()

In [ ]:
df.head(500)

In [ ]:
df.head(100)

In [ ]:
# Structure Data
X_train = df[["returns", "range"]].iloc[:500]
X_test = df[["returns", "range"]].iloc[500:]
saved_df = df.iloc[500:]

print(f"Train Length: {len(X_train)}")
print(f"Test Length: {len(X_test)}")

print(f"X_train From: {X_train.head(1).index.item()}")
print(f"X_train To: {X_train.tail(1).index.item()}")
print(f"X_test From: {X_test.head(1).index.item()}")
print(f"X_test To: {X_test.tail(1).index.item()}")

saved_df.head(10)

# Train HMM

In [ ]:
# Fit model
hmm_model = GaussianHMM(n_components=4, covariance_type="full", n_iter=1000, random_state=42)
hmm_model.fit(X_train)
hmm_model.predict(X_train)

In [ ]:
# Make prediction on test data
df_main = saved_df.copy()
df_main.drop(columns=["high", "low"], inplace=True)

hmm_results = hmm_model.predict(X_test)
df_main["hmm"] = hmm_results
df_main.head(100)

# Run Backtest

In [ ]:
# Add MA signals
df_main.loc[df_main["ma_12"] > df_main["ma_21"], "ma_signal"] = 1
df_main.loc[df_main["ma_12"] <= df_main["ma_21"], "ma_signal"] = 0

In [ ]:
# Add HMM signals
favorable_states = [0, 3]
hmm_values = df_main["hmm"].values
hmm_values = [1 if x in favorable_states else 0 for x in hmm_values]
df_main["hmm_signal"] = hmm_values

In [ ]:
# Add combined signal
df_main["main_signal"] = 0
df_main.loc[(df_main["ma_signal"] == 1) & (df_main["hmm_signal"] == 1), "main_signal"] = 1
df_main["main_signal"] = df_main["main_signal"].shift(1)

In [ ]:
# Benchmark returns
df_main["log_returns_benchmark"] = np.log(df_main["close"] / df_main["close"].shift(1))
df_main["benchmark_product"] = df_main["log_returns_benchmark"].cumsum()
df_main["benchmark_product_exp"] = np.exp(df_main["benchmark_product"]) - 1

In [ ]:
# Strategy returns
df_main["log_returns_strategy"] = np.log(df_main["open"].shift(-1) / df_main["open"]).squeeze() * df_main["main_signal"]
df_main["log_returns_product"] = df_main["log_returns_strategy"].cumsum()
df_main["strategy_product_exp"] = np.exp(df_main["log_returns_product"]) - 1

In [ ]:
# Review results table
df_main.dropna(inplace=True)
df_main.head(500)

# Calculate Metrics

In [ ]:
# Sharpe Ratio Function
def calculate_sharpe_ratio(returns: float) -> float:
    N = 365
    rf = 0.01
    SQRTN = np.sqrt(N)
    mean = returns.mean() * N
    sigma = returns.std() * SQRTN
    sharpe_ratio = round((mean - rf) / sigma, 3)
    return sharpe_ratio

In [ ]:
# Metrics
benchmark_returns = round(df_main["benchmark_product_exp"].values[-1] * 100, 1)
strategy_returns = round(df_main["strategy_product_exp"].values[-1] * 100, 1)

benchmark_sharpe = calculate_sharpe_ratio(df_main["log_returns_benchmark"].values)
strategy_sharpe = calculate_sharpe_ratio(df_main["log_returns_strategy"].values)

In [ ]:
print(f"Benchmark returns: {benchmark_returns}%")
print(f"Strategy returns: {strategy_returns}%")

print(f"Benchmark Sharpe: {benchmark_sharpe}")
print(f"Strategy Sharpe: {strategy_sharpe}")

# Plot Results

In [ ]:
# Plot Equity Curve
fig = plt.figure(figsize=(18, 10))
plt.plot(df_main["benchmark_product_exp"].values, color="orange")
plt.plot(df_main["strategy_product_exp"].values, color="green")
plt.legend(["benchmark (buy & hold)", "strategy (12 & 21 day SMA cross)"])
plt.show()

# Save Data

In [ ]:
df_main.to_csv("saved-data.csv", index=False)